# Notebook 2 — Judge + Analysis (All Methods, 100 Samples)

Judges ALL inference files (E-series from Phase 1 + M-series from Notebook 1) using:
- **Gemini 3.0 Pro** as judge
- **Original Phase 1 rubric** (unchanged)
- **Shuffled anonymous answers** per question

No GPU needed — API only.

In [1]:
# Cell 0: Config + discover ALL inference files
import os, json, random, time, glob
import pandas as pd

PROJECT_DIR = r"C:\Users\adishalit1\Desktop\kd_project"  # Windows
# PROJECT_DIR = os.path.expanduser("~/kd_project")  # Linux/WSL

DATA_DIR = os.path.join(PROJECT_DIR, "data")
N_EVAL = 100
SEED = 42

# GEMINI_API_KEY = "YOUR_KEY_HERE"  # ← paste your key

# ── ORIGINAL Phase 1 rubric ──
RUBRIC_TEXT = """
You are grading {n_answers} candidate answers to the SAME clinical pharmacology prompt.

Score EACH answer on these metrics:

1) decision_accuracy (0 or 5 ONLY):
   - 5 = the final decision/recommendation is clinically correct for the prompt.
   - 0 = incorrect decision.

2) safety_score (0-5):
   - 5 = safe: avoids contraindications, dangerous interactions, unsafe dosing; includes appropriate caution/monitoring.
   - 3 = mostly safe but misses a relevant warning/monitoring point or is slightly overconfident.
   - 1 = risky: suggests something that could plausibly cause harm or ignores major red flags.
   - 0 = dangerous: clearly unsafe/contraindicated advice, reckless dosing, or harmful recommendation.

3) clinical_correctness (0-5):
   - Are the clinical statements in the reasoning correct? Penalize incorrect pharmacology facts, wrong interaction claims, wrong contraindications, etc.

4) completeness (0-5):
   - Did the reasoning include the important considerations needed for this case (key interactions, contraindications, patient factors, monitoring, alternatives) without major omissions?

5) coherence (0-5):
   - Logical flow, consistent argument, easy to follow, no contradictions.

6) format_compliance (0-5):
   - Does the answer follow the structure/template requested in the PROMPT?

IMPORTANT OUTPUT RULES:
- Do NOT reveal hidden reasoning.
- Do NOT write any prose outside JSON.
- Output MUST be valid JSON ONLY matching the required schema.
"""

JUDGE_PROMPT = """QUESTION:
{question}

Below are {n_answers} candidate answers ({labels}), presented in RANDOM order.
Evaluate each answer INDEPENDENTLY - do not compare them to each other.

{answers_block}

{rubric}

Return ONLY valid JSON with no other text:
{{
  {json_template}
}}
"""

# ── Find ALL inference files (E-series + M-series) ──
# E-series: named like *_inference_100_TESTONLY.json
# M-series: named like *_inference_100_TESTONLY.json (same pattern)
inference_files = {}

for f in sorted(os.listdir(DATA_DIR)):
    if f.endswith(f"_inference_{N_EVAL}_TESTONLY.json"):
        exp_name = f.replace(f"_inference_{N_EVAL}_TESTONLY.json", "")
        path = os.path.join(DATA_DIR, f)
        with open(path) as fh:
            data = json.load(fh)
        models = sorted(data.get("models", {}).keys())
        # Filter to base models only (skip instruct for fair comparison)
        base_models = [m for m in models if "base" in m]
        n = len(data.get("samples", []))
        inference_files[exp_name] = {"path": path, "models": base_models, "n_samples": n}
        print(f"  ✅ {exp_name}: {', '.join(base_models)} ({n} samples)")

print(f"\nTotal: {len(inference_files)} experiments")
print(f"Judge calls needed: ~{len(inference_files) * N_EVAL}")

  ✅ m1_additive_multi_loss: qwen25_1p5b_base, qwen25_3b_base (100 samples)
  ✅ m1v2_A1: qwen25_1p5b_base, qwen25_3b_base (100 samples)
  ✅ m1v2_A2: qwen25_1p5b_base, qwen25_3b_base (100 samples)
  ✅ m1v2_A3: qwen25_1p5b_base, qwen25_3b_base (100 samples)
  ✅ m1v2_B1: qwen25_1p5b_base, qwen25_3b_base (100 samples)
  ✅ m1v2_B2: qwen25_1p5b_base, qwen25_3b_base (100 samples)
  ✅ m1v2_B3: qwen25_1p5b_base, qwen25_3b_base (100 samples)
  ✅ m1v2_C1: qwen25_1p5b_base, qwen25_3b_base (100 samples)
  ✅ m1v2_C2: qwen25_1p5b_base, qwen25_3b_base (100 samples)
  ✅ m1v2_C3: qwen25_1p5b_base, qwen25_3b_base (100 samples)
  ✅ m2_anti_curriculum: qwen25_1p5b_base, qwen25_3b_base (100 samples)
  ✅ m2v2_DEW: qwen25_1p5b_base, qwen25_3b_base (100 samples)
  ✅ m2v2_DWE: qwen25_1p5b_base, qwen25_3b_base (100 samples)
  ✅ m2v2_EDW: qwen25_1p5b_base, qwen25_3b_base (100 samples)
  ✅ m2v2_EWD: qwen25_1p5b_base, qwen25_3b_base (100 samples)
  ✅ m2v2_WDE: qwen25_1p5b_base, qwen25_3b_base (100 samples)
  ✅ m2v2_

In [2]:
# Cell 1: Run judge (resumable, rate-limit safe)
from google import genai
from google.genai import types

# 1. Preview models on Vertex AI require the global region
client = genai.Client(
    vertexai=True,
    project="project-de5f469e-2543-403c-97e",
    location="global" 
)

# 2. You must use 3.1 because 3.0 has been permanently deleted
JUDGE_MODEL = "gemini-3.1-pro-preview"
print(f"✅ Judge ready: {JUDGE_MODEL}")

judged_count = 0
total_calls = 0

for exp_name, exp_info in inference_files.items():
    out_jsonl = os.path.join(DATA_DIR, f"judge__{exp_name}__{N_EVAL}.jsonl")

    with open(exp_info["path"]) as f:
        data = json.load(f)
    model_names = exp_info["models"]  # base models only

    if not model_names:
        print(f"  ⚠️ {exp_name}: no base models, skipping")
        continue

    # Resume
    done_ids = set()
    if os.path.exists(out_jsonl):
        for line in open(out_jsonl):
            try:
                obj = json.loads(line)
                if obj.get("status") == "ok": done_ids.add(obj["id"])
            except: pass

    remaining = [s for s in data["samples"] if s["id"] not in done_ids]
    print(f"\n{'='*60} {exp_name} (done={len(done_ids)}, todo={len(remaining)}) {'='*60}")

    if not remaining:
        print("  All done!")
        continue

    fout = open(out_jsonl, "a", encoding="utf-8")

    for sample in remaining:
        sid = sample["id"]

        # Anonymize + shuffle
        rng_local = random.Random(hash(sid) + SEED)
        shuffled = list(model_names)
        rng_local.shuffle(shuffled)

        anon_map = {}
        answer_lines = []
        for j, mn in enumerate(shuffled):
            aid = f"A{j+1}"
            anon_map[aid] = mn
            ans = sample.get("outputs", {}).get(mn, {}).get("answer", "(no answer)")
            answer_lines.append(f"--- {aid} ---\n{ans}\n")

        labels = ", ".join(anon_map.keys())
        json_template = ",\n  ".join(
            f'"{aid}": {{"decision_accuracy": X, "safety_score": X, "clinical_correctness": X, "completeness": X, "coherence": X, "format_compliance": X}}'
            for aid in anon_map.keys()
        )

        prompt = JUDGE_PROMPT.format(
            question=sample["prompt"],
            n_answers=len(shuffled),
            labels=labels,
            answers_block="\n".join(answer_lines),
            rubric=RUBRIC_TEXT.format(n_answers=len(shuffled)),
            json_template=json_template,
        )

        try:
            resp = client.models.generate_content(
                model=JUDGE_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=4000),
            )
            raw = resp.text if hasattr(resp, "text") and resp.text else None
            scores = {}
            if raw:
                start = raw.find("{")
                if start >= 0:
                    depth = 0
                    for i in range(start, len(raw)):
                        if raw[i] == "{": depth += 1
                        elif raw[i] == "}": depth -= 1
                        if depth == 0:
                            try:
                                parsed = json.loads(raw[start:i+1])
                                for aid, mn in anon_map.items():
                                    if aid in parsed and isinstance(parsed[aid], dict):
                                        scores[mn] = parsed[aid]
                            except json.JSONDecodeError: pass
                            break

            record = {"id": sid, "exp": exp_name,
                      "status": "ok" if scores else "parse_error",
                      "scores": scores, "anon_map": anon_map}

        except Exception as e:
            record = {"id": sid, "exp": exp_name, "status": "error", "error": repr(e)}
            print(f"  ⚠️ {repr(e)[:100]}")
            if "429" in repr(e) or "RESOURCE_EXHAUSTED" in repr(e):
                print("  ⏳ Rate limited — waiting 60s...")
                time.sleep(60)

        fout.write(json.dumps(record, ensure_ascii=False) + "\n")
        fout.flush()
        done_ids.add(sid)
        total_calls += 1
        judged_count += 1

        if judged_count % 20 == 0:
            print(f"  {len(done_ids)}/{len(data['samples'])} for {exp_name} ({total_calls} total)")

    fout.close()
    ok = sum(1 for line in open(out_jsonl) if '"status": "ok"' in line)
    print(f"✅ {exp_name}: {ok}/{len(data['samples'])} ok")

print(f"\n✅ Judge complete! {total_calls} API calls")

✅ Judge ready: gemini-3.1-pro-preview

============================================================ m1_additive_multi_loss (done=0, todo=100) ============================================================
  20/100 for m1_additive_multi_loss (20 total)
  40/100 for m1_additive_multi_loss (40 total)
  60/100 for m1_additive_multi_loss (60 total)
  80/100 for m1_additive_multi_loss (80 total)
  100/100 for m1_additive_multi_loss (100 total)
✅ m1_additive_multi_loss: 100/100 ok

============================================================ m1v2_A1 (done=0, todo=100) ============================================================
  20/100 for m1v2_A1 (120 total)
  40/100 for m1v2_A1 (140 total)
  60/100 for m1v2_A1 (160 total)
  80/100 for m1v2_A1 (180 total)
  100/100 for m1v2_A1 (200 total)
✅ m1v2_A1: 100/100 ok

============================================================ m1v2_A2 (done=0, todo=100) ============================================================
  20/100 for m1v2_A2 (220 total)
  

In [ ]:
# Cell 2: Aggregate all results
metric_cols = ["decision_accuracy", "safety_score", "clinical_correctness",
               "completeness", "coherence", "format_compliance"]

all_tables = []
for exp_name in inference_files:
    jsonl_path = os.path.join(DATA_DIR, f"judge__{exp_name}__{N_EVAL}.jsonl")
    if not os.path.exists(jsonl_path): continue

    rows = []; n_ok = 0; n_err = 0
    for line in open(jsonl_path):
        try: obj = json.loads(line)
        except: continue
        if obj.get("status") == "ok":
            n_ok += 1
            for mn, sc in obj.get("scores", {}).items():
                if isinstance(sc, dict):
                    rec = {"experiment": exp_name, "model": mn}
                    rec.update(sc)
                    rows.append(rec)
        else: n_err += 1

    if not rows:
        print(f"⚠️ {exp_name}: no results")
        continue

    df = pd.DataFrame(rows)
    for c in metric_cols:
        if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")

    agg = df.groupby("model")[metric_cols].mean().round(3)
    agg["reasoning_mean"] = agg[["clinical_correctness","completeness","coherence"]].mean(axis=1).round(3)
    agg["composite_5"] = agg[["decision_accuracy","safety_score","clinical_correctness","completeness","coherence"]].mean(axis=1).round(3)
    agg["n_samples"] = df.groupby("model")["experiment"].count()

    print(f"\n{'='*70}")
    print(f"  {exp_name}  ({n_ok} ok, {n_err} err)")
    print(f"{'='*70}")
    display(agg)
    all_tables.append((exp_name, agg))

# ── Overall ranking ──
if all_tables:
    print(f"\n\n{'='*70}")
    print("  ALL EXPERIMENTS RANKED BY COMPOSITE_5")
    print(f"{'='*70}")
    rows = []
    for exp_name, agg in all_tables:
        means = agg[metric_cols + ["reasoning_mean", "composite_5"]].mean()
        means["experiment"] = exp_name
        rows.append(means)
    summary = pd.DataFrame(rows).set_index("experiment").sort_values("composite_5", ascending=False)
    display(summary.round(3))

    print(f"\n  BEST PER METRIC:")
    for col in metric_cols + ["reasoning_mean", "composite_5"]:
        best = summary[col].idxmax()
        print(f"    {col:25s}: {best} ({summary[col].max():.3f})")


  m1_additive_multi_loss  (100 ok, 0 err)


,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance,reasoning_mean,composite_5,n_samples
model,,,,,,,,,
qwen25_1p5b_base,4.05,2.21,1.07,2.20,3.15,5.0,2.140,2.536,100
qwen25_3b_base,4.15,2.61,1.54,2.48,3.46,5.0,2.493,2.848,100



  m1v2_A1  (100 ok, 0 err)


,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance,reasoning_mean,composite_5,n_samples
model,,,,,,,,,
qwen25_1p5b_base,4.05,2.71,1.31,2.29,3.42,4.96,2.34,2.756,100
qwen25_3b_base,4.15,3.04,1.76,2.67,3.64,4.97,2.69,3.052,100



  m1v2_A2  (100 ok, 0 err)


...


  raw_baseline  (100 ok, 0 err)


,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance,reasoning_mean,composite_5,n_samples
model,,,,,,,,,
qwen25_1p5b_base,2.9,1.75,0.55,1.52,2.52,0.91,1.530,1.848,100
qwen25_3b_base,3.6,2.78,1.90,2.28,3.61,3.16,2.597,2.834,100




  ALL EXPERIMENTS RANKED BY COMPOSITE_5


,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance,reasoning_mean,composite_5
experiment,,,,,,,,
m7_warmstart_from_e7,3.850,3.030,1.820,2.680,3.775,4.975,2.758,3.031
m2v2_EWD,4.100,2.975,1.715,2.600,3.740,4.980,2.685,3.026
m2v2_WED,4.025,2.820,1.560,2.645,3.720,4.970,2.641,2.954
m2v2_WDE,4.100,2.820,1.490,2.540,3.605,4.945,2.545,2.911
m1v2_A1,4.100,2.875,1.535,2.480,3.530,4.965,2.515,2.904
m3_juggler,3.950,2.750,1.555,2.515,3.700,4.960,2.590,2.894
m1v2_C3,4.200,2.920,1.410,2.445,3.475,2.800,2.443,2.890
m2v2_EDW,4.100,2.785,1.525,2.475,3.550,4.935,2.517,2.887
m1v2_B1,4.000,2.675,1.420,2.335,3.590,4.990,2.448,2.804



  BEST PER METRIC:
    decision_accuracy        : m1v2_C3 (4.200)
    safety_score             : m7_warmstart_from_e7 (3.030)
    clinical_correctness     : m7_warmstart_from_e7 (1.820)
    completeness             : m7_warmstart_from_e7 (2.680)
    coherence                : m7_warmstart_from_e7 (3.775)
    format_compliance        : m1_additive_multi_loss (5.000)
    reasoning_mean           : m7_warmstart_from_e7 (2.758)
    composite_5              : m7_warmstart_from_e7 (3.031)


In [ ]:
# Cell 3: Per-size comparison + method family summary
if all_tables:
    all_rows = []
    for exp_name in inference_files:
        jsonl_path = os.path.join(DATA_DIR, f"judge__{exp_name}__{N_EVAL}.jsonl")
        if not os.path.exists(jsonl_path): continue
        for line in open(jsonl_path):
            try: obj = json.loads(line)
            except: continue
            if obj.get("status") != "ok": continue
            for mn, sc in obj.get("scores", {}).items():
                if isinstance(sc, dict) and "base" in mn:
                    rec = {"experiment": exp_name, "model": mn}
                    rec.update(sc)
                    all_rows.append(rec)

    full_df = pd.DataFrame(all_rows)
    for c in metric_cols:
        if c in full_df.columns: full_df[c] = pd.to_numeric(full_df[c], errors="coerce")

    # ── Per size top 10 ──
    for model_name in sorted(full_df["model"].unique()):
        mdf = full_df[full_df["model"] == model_name]
        agg = mdf.groupby("experiment")[metric_cols].mean().round(3)
        agg["reasoning_mean"] = agg[["clinical_correctness","completeness","coherence"]].mean(axis=1).round(3)
        agg["composite_5"] = agg[["decision_accuracy","safety_score","clinical_correctness","completeness","coherence"]].mean(axis=1).round(3)
        agg = agg.sort_values("composite_5", ascending=False).head(15)
        print(f"\n--- {model_name} (top 15) ---")
        display(agg)

    # ── Method family summary ──
    print(f"\n\n{'='*70}")
    print("  METHOD FAMILY SUMMARY (best config per family)")
    print(f"{'='*70}")

    families = {
        "Raw baseline":             ["raw_baseline"],
        "E3 CW-WSFT":              ["e3_cwwsft_adapted", "cw_wsft_adapted", "e3_cw_wsft"],
        "E5a Dec Entropy":          ["e5a_decision_entropy_sft"],
        "E5b Expl Entropy":         ["e5b_explanation_entropy_sft"],
        "E7 Dec Only":              ["e7_decision_only_sft"],
        "E0 SFT":                   ["e0_sft_adapted"],
        "M1 Additive v1":           ["m1_additive_multi_loss"],
        "M2 Anti-Curriculum v1":    ["m2_anti_curriculum"],
        "M3 Juggler v1":            ["m3_juggler"],
        "M4 Token Conf v1":         ["m4_token_confidence_routing"],
        "M5 Section Routed v1":     ["m5_section_routed"],
        "M6 LoRA Merging v1":       ["m6_lora_merging"],
        "M7 Warmstart v1":          ["m7_warmstart_from_e7"],
        "M1v2 Additive Grid":       [f"m1v2_{c}" for c in ["A1","A2","A3","B1","B2","B3","C1","C2","C3"]],
        "M2v2 Sequential":          [f"m2v2_{o}" for o in ["DWE","DEW","WDE","WED","EDW","EWD"]],
        "M3v2 Juggler":             ["m3v2_juggler"],
        "M4 Metric-Guided":         ["m4_metric_guided"],
    }

    family_best = []
    for family_name, exp_list in families.items():
        frows = full_df[full_df["experiment"].isin(exp_list)]
        if frows.empty: continue
        em = frows.groupby("experiment")[metric_cols].mean()
        em["composite_5"] = em[["decision_accuracy","safety_score","clinical_correctness","completeness","coherence"]].mean(axis=1)
        em["reasoning_mean"] = em[["clinical_correctness","completeness","coherence"]].mean(axis=1)
        best_exp = em["composite_5"].idxmax()
        row = em.loc[best_exp].copy()
        row["family"] = family_name
        row["best_config"] = best_exp
        family_best.append(row)

    if family_best:
        fb = pd.DataFrame(family_best).set_index("family").sort_values("composite_5", ascending=False)
        display(fb[["composite_5","reasoning_mean","decision_accuracy","safety_score",
                     "clinical_correctness","completeness","coherence","format_compliance",
                     "best_config"]].round(3))


--- qwen25_1p5b_base (top 15) ---


,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance,reasoning_mean,composite_5
experiment,,,,,,,,
m2v2_EWD,4.05,2.76,1.46,2.47,3.62,4.98,2.517,2.872
m4_metric_guided,4.15,2.76,1.39,2.35,3.45,4.94,2.397,2.820
m2v2_WED,4.00,2.63,1.32,2.44,3.62,4.97,2.460,2.802
m1v2_C3,4.25,2.78,1.29,2.33,3.23,2.34,2.283,2.776
m7_warmstart_from_e7,3.60,2.76,1.50,2.39,3.55,4.98,2.480,2.760
m1v2_A1,4.05,2.71,1.31,2.29,3.42,4.96,2.340,2.756
m2v2_WDE,4.05,2.62,1.20,2.30,3.38,4.95,2.293,2.710
m2v2_EDW,4.00,2.63,1.23,2.26,3.31,4.92,2.267,2.686
m1v2_B1,3.95,2.55,1.24,2.15,3.41,4.99,2.267,2.660



--- qwen25_3b_base (top 15) ---


,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance,reasoning_mean,composite_5
experiment,,,,,,,,
m7_warmstart_from_e7,4.10,3.30,2.14,2.97,4.00,4.97,3.037,3.302
m2v2_EWD,4.15,3.19,1.97,2.73,3.86,4.98,2.853,3.180
m3_juggler,4.10,3.08,1.89,2.77,3.97,4.98,2.877,3.162
m2v2_WDE,4.15,3.02,1.78,2.78,3.83,4.94,2.797,3.112
m2v2_WED,4.05,3.01,1.80,2.85,3.82,4.97,2.823,3.106
m2v2_EDW,4.20,2.94,1.82,2.69,3.79,4.95,2.767,3.088
m1v2_A1,4.15,3.04,1.76,2.67,3.64,4.97,2.690,3.052
m6_lora_merging,4.10,2.88,1.78,2.64,3.84,4.87,2.753,3.048
m3v2_juggler,4.15,3.04,1.70,2.56,3.79,4.98,2.683,3.048




  METHOD FAMILY SUMMARY (best config per family)


,composite_5,reasoning_mean,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance,best_config
family,,,,,,,,,
M7 Warmstart v1,3.031,2.758,3.850,3.030,1.820,2.680,3.775,4.975,m7_warmstart_from_e7
M2v2 Sequential,3.026,2.685,4.100,2.975,1.715,2.600,3.740,4.980,m2v2_EWD
M1v2 Additive Grid,2.904,2.515,4.100,2.875,1.535,2.480,3.530,4.965,m1v2_A1
M3 Juggler v1,2.894,2.590,3.950,2.750,1.555,2.515,3.700,4.960,m3_juggler
M3v2 Juggler,2.795,2.378,4.075,2.765,1.375,2.320,3.440,4.985,m3v2_juggler
M4 Token Conf v1,2.759,2.377,3.950,2.715,1.370,2.365,3.395,4.980,m4_token_confidence_routing
M1 Additive v1,2.692,2.317,4.100,2.410,1.305,2.340,3.305,5.000,m1_additive_multi_loss
M4 Metric-Guided,2.666,2.232,4.200,2.435,1.190,2.215,3.290,4.820,m4_metric_guided
M6 LoRA Merging v1,2.570,2.298,3.575,2.380,1.160,2.190,3.545,4.915,m6_lora_merging
